# Create a Genie Space for Product Manual Comparison

This notebook creates a [Genie Space](https://docs.databricks.com/en/genie/) that lets users ask natural-language questions about **cordless drill/driver specifications** extracted from manufacturer PDF manuals.

The space is backed by ``{catalog}.{schema}.{table_prefix}_productmanuals_processed``. Widget defaults match ``databricks_etl/databricks.yml`` (including ``warehouse_id`` for the SQL warehouse used when creating the space).

**Example questions a user can ask:**
- "Which drill has the highest torque?"
- "Compare the weight of all 18V drills"
- "Which tool is compatible with ProCORE batteries?"

In [0]:
# %pip install databricks-sdk -q

from databricks.sdk import WorkspaceClient
from manage_genie_space import *
import json
import uuid

w = WorkspaceClient()

# Defaults match databricks_etl/databricks.yml (catalog, schema, table_prefix, warehouse_id)
dbutils.widgets.text("catalog", "<CATALOG>")
dbutils.widgets.text("schema", "<SCHEMA>")
dbutils.widgets.text("table_prefix", "<TABLE_PREFIX>")
dbutils.widgets.text("warehouse_id", "<SQL_WAREHOUSE_ID>")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_prefix = dbutils.widgets.get("table_prefix")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")

TABLE_FULL_NAME = f"{catalog}.{schema}.{table_prefix}_productmanuals_processed"

print(f"TABLE_FULL_NAME={TABLE_FULL_NAME}")

GENIE_SPACE_TITLE = f"{table_prefix}_Power Tool Product Comparison"
DESCRIPTION = (
    "Compare cordless drill/driver specifications across manufacturers. "
    "Ask about voltage, torque, speed, weight, battery compatibility, and more. "
    f"Data source: {TABLE_FULL_NAME}"
)

SAMPLE_QUESTIONS = [
    "Which drill has the highest max torque?",
    "Compare all drills by weight and voltage",
    "List drills compatible with 18V batteries",
    "Which manufacturer offers the fastest high-speed RPM?",
    "Show me all specifications for the DeWalt DCD991",
]

SAMPLE_QUESTIONS_UUID = [
    {"id": uuid.uuid4().hex, "question": [q]} for q in SAMPLE_QUESTIONS
]

SERIALIZED_SPACE = json.dumps({
    "version": 2,
    "config": {
        "sample_questions": SAMPLE_QUESTIONS_UUID
    },
    "data_sources": {
        "tables": [
            {
                "identifier": TABLE_FULL_NAME,
                "column_configs": [
                    {"column_name": "compatible_batteries", "enable_format_assistance": True},
                    {"column_name": "compatible_chargers", "enable_format_assistance": True},
                    {"column_name": "manufacturer", "enable_format_assistance": True, "enable_entity_matching": True},
                    {"column_name": "max_torque_nm", "enable_format_assistance": True},
                    {"column_name": "model_number", "enable_format_assistance": True, "enable_entity_matching": True},
                    {"column_name": "no_load_speed_high_rpm", "enable_format_assistance": True},
                    {"column_name": "no_load_speed_low_rpm", "enable_format_assistance": True},
                    {"column_name": "product_name", "enable_format_assistance": True},
                    {"column_name": "product_type", "enable_format_assistance": True, "enable_entity_matching": True},
                    {"column_name": "rated_voltage_v", "enable_format_assistance": True},
                    {"column_name": "weight_kg", "enable_format_assistance": True},
                ]
            }
        ]
    }
})

In [0]:
genie_space = get_existing_genie_space(w, GENIE_SPACE_TITLE)

In [0]:
if not genie_space:
    response = create_genie_space(
        w, WAREHOUSE_ID, SERIALIZED_SPACE, GENIE_SPACE_TITLE, DESCRIPTION
    )
    print_space_info(response, created=True)
else:
    print_space_info(genie_space)